# Generic Lax series from only $\mathbf A^{(0)}$

This notebook is the next stage of the nonparaxial-beam prototype.

The essential change is that the user now supplies only the zeroth-order paraxial vector-potential envelope

$$
\mathbf A^{(0)}(X,Y,Z),
$$

and the code generates

$$
\mathbf A
=
\mathbf A^{(0)}
+\epsilon^2\mathbf A^{(2)}
+\epsilon^4\mathbf A^{(4)}
+\cdots
$$

automatically.

The notebook then constructs the magnetic field from the generated vector potential.

## 1. The normalized wave equation

After removing the carrier $e^{ikz-i\omega t}$, every Cartesian component of the vector-potential envelope satisfies

$$
\left(
\Delta_\perp
+4i\partial_Z
+\epsilon^2\partial_Z^2
\right)A_j=0,
$$

where

$$
\Delta_\perp=\partial_X^2+\partial_Y^2.
$$

The zeroth-order field therefore obeys the paraxial equation

$$
\left(
\Delta_\perp+4i\partial_Z
\right)A_j^{(0)}=0.
$$

## 2. A boundary convention is still needed

The higher-order Lax equations do not uniquely determine $A^{(2)},A^{(4)},\ldots$, because a homogeneous paraxial solution can be added at every order.

For this generic constructor we choose the forward Helmholtz continuation that preserves the supplied field at the reference plane:

$$
\boxed{
A^{(2j)}(X,Y,0)=0,
\qquad j>0.
}
$$

Thus the input $A^{(0)}(X,Y,0)$ is kept exactly at $Z=0$.

This is a different higher-order boundary convention from the particular Salamin solution used in the earlier PRL benchmark. The curl machinery is common to both; the generated higher-order homogeneous pieces are not.

## 3. Imports

In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

from lax_series import (
    lax_operator_series,
    lax_expand_scalar,
    lax_expand_vector,
    lax_coefficients,
    paraxial_residual,
    wave_residual,
    truncate,
    transverse_laplacian,
    magnetic_field_from_envelope,
)

sp.init_printing()

I = sp.I
X, Y, Z = sp.symbols("X Y Z", real=True)
eps, k, Aamp = sp.symbols("eps k Aamp", positive=True, real=True)

## 4. The generic propagation operator

It is useful to see what the code is doing before applying it to a particular beam.

Let $q$ stand for the transverse Laplacian $\Delta_\perp$. The exact forward Helmholtz propagation can be expanded relative to paraxial propagation. The resulting correction operator is a polynomial in $q$ and $\epsilon$.

Through fourth order it has the structure

$$
\mathcal R
=
1
-\frac{iZ\epsilon^2}{64}q^2
+\epsilon^4
\left(
\frac{iZ}{512}q^3
-\frac{Z^2}{8192}q^4
\right)
+O(\epsilon^6).
$$

The generic generator replaces each $q^n$ by $\Delta_\perp^n$ acting on the supplied paraxial seed.

In [ ]:
Rop, q = lax_operator_series(eps, Z, order=4)
display(Rop)

Therefore, for any valid paraxial seed $A^{(0)}$,

$$
A^{(2)}
=-\frac{iZ}{64}\Delta_\perp^2 A^{(0)},
$$

and

$$
A^{(4)}
=
\frac{iZ}{512}\Delta_\perp^3 A^{(0)}
-\frac{Z^2}{8192}\Delta_\perp^4 A^{(0)}.
$$

These are operator formulas: they do not assume a Gaussian beam.

## 5. A simple exact validation: one transverse Fourier mode

Before using a Gaussian, test the generator on a paraxial transverse Fourier mode

$$
a^{(0)}
=
\exp\left[
 i pX+i qY-rac{i}{4}(p^2+q^2)Z
\right].
$$

This seed is especially useful because every transverse derivative simply returns a power of the transverse wave number.

In [ ]:
p, qy = sp.symbols("p q_y", real=True)
p2 = p**2 + qy**2

plane_seed = sp.exp(
    I*p*X + I*qy*Y - I*p2*Z/4
)

paraxial_check = sp.simplify(
    paraxial_residual(plane_seed, (X, Y, Z))
)

print("paraxial residual =", paraxial_check)

In [ ]:
plane_lax = lax_expand_scalar(
    plane_seed,
    coords=(X, Y, Z),
    eps=eps,
    order=6,
)

plane_wave_residual = sp.simplify(
    truncate(
        wave_residual(plane_lax, (X, Y, Z), eps),
        eps,
        order=6,
    )
)

print("full wave-equation residual through O(eps^6) =", plane_wave_residual)

A zero result in both cells checks two different things:

1. the supplied field is a valid paraxial seed;
2. the generated Lax series satisfies the full normalized wave equation through the requested order.

## 6. Gaussian example: define only $\mathbf A^{(0)}$

Now use the familiar paraxial Gaussian

$$
f(Z)=\frac{1}{1+iZ},
$$

$$
\psi_0
=
f\exp\left[-f(X^2+Y^2)\right].
$$

For the carrier convention used by `lax_series.py`, $e^{ikz-i\omega t}$, this is the Gaussian with the correct propagation sign. For this demonstration choose a longitudinal vector potential,

$$
\boxed{
\mathbf A^{(0)}
=
(0,0,A_{\rm amp}\psi_0).
}
$$

`Aamp` is only an overall scalar amplitude. `A0` below is the actual zeroth-order vector field.


### Phasor-sign convention

The earlier Salamin/PRL notebook used the carrier convention

$$
e^{i\omega t-ikz},
$$

for which the Gaussian factor was

$$
f_{\rm PRL}=\frac{1}{1-iZ}.
$$

The generic backend in `lax_series.py` currently uses the conjugate convention

$$
e^{ikz-i\omega t}.
$$

Therefore the corresponding paraxial Gaussian is

$$
\boxed{
f=\frac{1}{1+iZ}.
}
$$

The two descriptions represent the same real physics after consistent complex conjugation. The important rule is simply not to mix the carrier sign and the Gaussian sign convention.


In [ ]:
rho2 = X**2 + Y**2
f = 1 / (1 + I*Z)
psi0 = f * sp.exp(-f * rho2)

A0 = (
    sp.Integer(0),
    sp.Integer(0),
    Aamp * psi0,
)

A0

First check that the supplied Gaussian seed really satisfies the paraxial equation.

In [ ]:
gaussian_paraxial_residual = sp.simplify(
    paraxial_residual(psi0, (X, Y, Z))
)

print("Gaussian paraxial residual =", gaussian_paraxial_residual)

## 7. Generate $A^{(2)}$ and $A^{(4)}$ automatically

There are no hand-written `psi2` or `psi4` expressions here. The only beam-specific input is `A0`.

In [ ]:
A = lax_expand_vector(
    A0,
    coords=(X, Y, Z),
    eps=eps,
    order=4,
)

Ax, Ay, Az = A
Az_coefficients = lax_coefficients(Az, eps, order=4)

print("generated orders:", sorted(Az_coefficients))

We can verify explicitly that the generated coefficients are the generic operator expressions shown above.

In [ ]:
def lap(expr):
    return transverse_laplacian(expr, X, Y)

A0z = A0[2]

lap2 = lap(lap(A0z))
lap3 = lap(lap2)
lap4 = lap(lap3)

A2_operator = -I * Z * lap2 / 64
A4_operator = I * Z * lap3 / 512 - Z**2 * lap4 / 8192

print("A^(2) difference =", sp.simplify(Az_coefficients[2] - A2_operator))
print("A^(4) difference =", sp.simplify(Az_coefficients[4] - A4_operator))

## 8. Check the reference-plane convention

Because of the chosen forward-continuation convention, all generated corrections vanish at $Z=0$.

In [ ]:
for n in (2, 4):
    value = sp.simplify(Az_coefficients[n].subs(Z, 0))
    print(f"A^({n})(Z=0) =", value)

This is the key distinction from the earlier Salamin benchmark: the generic constructor keeps the supplied focal/reference-plane field fixed, whereas Salamin's particular higher-order solution includes nonzero homogeneous corrections at the focus.

## 9. Visualize the generated correction to the vector potential

At $Z=0$ the correction is zero by construction, so look one Rayleigh length away, $Z=1$.

The following plot compares the magnitude of the paraxial longitudinal potential with the potential including the generated $O(\epsilon^2)$ correction.

In [ ]:
Az_order2 = lax_expand_scalar(
    A0z,
    coords=(X, Y, Z),
    eps=eps,
    order=2,
)

Az0_fn = sp.lambdify((X, Z, Aamp), A0z.subs(Y, 0), modules="numpy")
Az2_fn = sp.lambdify((X, Z, eps, Aamp), Az_order2.subs(Y, 0), modules="numpy")

x_values = np.linspace(-2.5, 2.5, 400)
z_value = 1.0
eps_value = 0.35

az0_values = Az0_fn(x_values, z_value, 1.0)
az2_values = Az2_fn(x_values, z_value, eps_value, 1.0)

plt.figure(figsize=(7, 4.5))
plt.plot(x_values, np.abs(az0_values), label=r"$|A_z^{(0)}|$")
plt.plot(x_values, np.abs(az2_values), "--", label=r"$|A_z^{(0)}+\epsilon^2A_z^{(2)}|$")
plt.xlabel(r"$X=x/w_0$")
plt.ylabel("magnitude")
plt.title(rf"Generated Lax correction at $Z=1$, $\epsilon={eps_value}$")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 10. Construct the magnetic field

For the full vector potential

$$
\mathbf A_{\rm full}
=
\mathbf A(X,Y,Z)e^{ikz-i\omega t},
$$

the magnetic-field envelope is

$$
\boxed{
\mathbf B
=
\nabla\times\mathbf A
+ik\,\hat{\mathbf z}\times\mathbf A.
}
$$

The second term comes from differentiating the carrier $e^{ikz}$ and is essential for a generic transverse vector potential.

For the present longitudinal Gaussian, $\hat{\mathbf z}\times\mathbf A=0$, so only the envelope curl contributes.

To keep the symbolic expressions compact in this walkthrough, construct $\mathbf A$ through $O(\epsilon^2)$ and therefore $\mathbf B$ through $O(\epsilon^3)$.

In [ ]:
A_for_B = lax_expand_vector(
    A0,
    coords=(X, Y, Z),
    eps=eps,
    order=2,
)

derivative_scales = (
    k * eps / 2,
    k * eps / 2,
    k * eps**2 / 2,
)

B = magnetic_field_from_envelope(
    A_for_B,
    coords=(X, Y, Z),
    eps=eps,
    order=3,
    k=k,
    derivative_scales=derivative_scales,
)

Bx, By, Bz = B
print("Bz =", Bz)

For an axisymmetric longitudinal vector potential, the magnetic field is azimuthal. On the positive $X$ axis ($Y=0$, $X>0$),

$$
B_\theta=B_y.
$$

In [ ]:
R = sp.symbols("R", nonnegative=True, real=True)

Btheta_axis = sp.factor(
    (By / (k*Aamp)).subs({X: R, Y: 0})
)

display(Btheta_axis)

## 11. Why the carrier term matters for a general $\mathbf A^{(0)}$

The longitudinal Gaussian does not exercise the carrier contribution, so consider the simple transverse seed

$$
\mathbf A^{(0)}=(A_{\rm amp},0,0).
$$

Its envelope curl is zero, but the full field contains $e^{ikz}$. Therefore

$$
\mathbf B
=ik\hat{\mathbf z}\times\mathbf A
=(0,ikA_{\rm amp},0).
$$

In [ ]:
A0_transverse = (Aamp, 0, 0)

A_transverse = lax_expand_vector(
    A0_transverse,
    coords=(X, Y, Z),
    eps=eps,
    order=4,
)

B_transverse = magnetic_field_from_envelope(
    A_transverse,
    coords=(X, Y, Z),
    eps=eps,
    order=4,
    k=k,
)

B_transverse

## 12. Minimal generic interface

For another paraxial beam, the intended workflow is now simply

```python
A0 = (Ax0, Ay0, Az0)

A = lax_expand_vector(
    A0,
    coords=(X, Y, Z),
    eps=eps,
    order=N,
)

B = magnetic_field_from_envelope(
    A,
    coords=(X, Y, Z),
    eps=eps,
    order=N_B,
    k=k,
    derivative_scales=(
        k*eps/2,
        k*eps/2,
        k*eps**2/2,
    ),
)
```

The beam-specific input is only $\mathbf A^{(0)}$ and its physical parameters. The Lax coefficients are generated by the common backend.

## 13. What to extend next

The current generator is generic in beam shape, but it uses one fixed higher-order boundary convention:

$$
A^{(2j)}(X,Y,0)=0.
$$

The natural next extension is to allow optional homogeneous higher-order pieces. That would let the same backend reproduce conventions such as the Salamin solution while retaining the generic forward-continuation case as the default.